In [26]:
# ======================= CONFIG =======================
RECORDING_DIR   = "/Users/cyrusjackson/Documents/Dev/Repos/SwarmRoboticsPerception/Assets/SimulationRecordings/Combinations/20260908_013309_flocking_40"

EPS             = 1.5      # DBSCAN eps  <-- change me (below ~0.6 every agent is its own group)
MIN_SAMPLES     = 3        # 1 = singletons count as their own group

START_TIME      = 1.0      # seconds -> "groups at starting"
GATE_MARGIN     = 0.0      # widen gates: x1 = wallLeft - margin, x2 = wallRight + margin
CROSS_QUANTILE  = 0.50     # 0.5 = median agent crosses the gate

OBSTACLE_NAME   = "WallObstacle"
N_JOBS          = -1       # parallel workers (-1 = all cores)
CACHE_PATH      = "dbscan_gate_counts_eps{eps}_ms{ms}.csv"   # cache key must include MIN_SAMPLES
USE_CACHE       = True
# ======================================================

In [27]:
import os, json, glob, warnings
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN
from joblib import Parallel, delayed

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 250)
pd.set_option("display.width", 200)

In [28]:
def n_clusters(xy, eps=EPS, min_samples=MIN_SAMPLES):
    if xy.shape[0] == 0:
        return np.nan
    labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(xy)
    n = len(set(labels) - {-1})
    n += int((labels == -1).sum())          # noise points counted as singletons
    return n


def first_frame_past(xq, gate):
    idx = np.nonzero(xq >= gate)[0]
    return int(idx[0]) if idx.size else None


def process_run(path, eps=EPS, min_samples=MIN_SAMPLES,
                start_time=START_TIME, gate_margin=GATE_MARGIN,
                cross_q=CROSS_QUANTILE, obstacle_name=OBSTACLE_NAME):
    with open(path) as fh:
        d = json.load(fh)

    h, frames = d["header"], d["frames"]
    n_agents  = int(h["agentCount"])

    pos = np.array([f["v"] for f in frames], dtype=np.float32).reshape(len(frames), n_agents, 5)[:, :, :2]
    t   = np.array([f["t"] for f in frames], dtype=np.float32)

    obs = next((g for g in h["geometry"] if g["name"] == obstacle_name), None)
    if obs is None:
        x1 = x2 = np.nan
    else:
        x1 = obs["x"] - obs["width"] / 2.0 - gate_margin
        x2 = obs["x"] + obs["width"] / 2.0 + gate_margin

    xq = np.quantile(pos[:, :, 0], cross_q, axis=1)

    i_start = int(np.argmin(np.abs(t - start_time)))
    i_end   = len(frames) - 1
    i_x1    = first_frame_past(xq, x1)
    i_x2    = first_frame_past(xq, x2)

    row = {
        "file":            os.path.basename(path),
        "PerceptionValue": round(float(h["perceptionRadius"]), 4),
        "MaxSpeed":        round(float(h["maxSpeed"]), 4),
        "Layout":          h.get("spawnLayoutId", ""),
        "eps":             eps,
        "x1":              x1,
        "x2":              x2,
        "t_start":         float(t[i_start]),
        "t_x1":            float(t[i_x1]) if i_x1 is not None else np.nan,
        "t_x2":            float(t[i_x2]) if i_x2 is not None else np.nan,
        "t_end":           float(t[i_end]),
        "GroupsStart":     n_clusters(pos[i_start], eps, min_samples),
        "GroupsX1":        n_clusters(pos[i_x1], eps, min_samples) if i_x1 is not None else np.nan,
        "GroupsX2":        n_clusters(pos[i_x2], eps, min_samples) if i_x2 is not None else np.nan,
        "GroupsEnd":       n_clusters(pos[i_end], eps, min_samples),
        "endReason":       h.get("endReason", ""),
    }
    return row

In [29]:
files = sorted(f for f in glob.glob(os.path.join(RECORDING_DIR, "*.json"))
               if not os.path.basename(f).startswith("batch_config"))
cache = CACHE_PATH.format(eps=EPS, ms=MIN_SAMPLES)

if USE_CACHE and os.path.exists(cache):
    runs = pd.read_csv(cache)
else:
    rows = Parallel(n_jobs=N_JOBS, verbose=5)(delayed(process_run)(f) for f in files)
    runs = pd.DataFrame(rows)
    runs.to_csv(cache, index=False)

len(files), runs.shape

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done  84 tasks      | elapsed:    0.7s
[Parallel(n_jobs=-1)]: Done 264 tasks      | elapsed:    1.8s
[Parallel(n_jobs=-1)]: Done 516 tasks      | elapsed:    3.2s
[Parallel(n_jobs=-1)]: Done 720 out of 720 | elapsed:    4.3s finished


(720, (720, 16))

In [30]:
per_run = runs[["PerceptionValue", "MaxSpeed", "Layout",
                "GroupsStart", "GroupsX1", "GroupsX2", "GroupsEnd"]] \
            .sort_values(["PerceptionValue", "MaxSpeed", "Layout"]) \
            .reset_index(drop=True)
per_run

,PerceptionValue,MaxSpeed,Layout,GroupsStart,GroupsX1,GroupsX2,GroupsEnd
0,0.5,1.5,20260908_013309_layout_00,3,4,4,7
1,0.5,1.5,20260908_013309_layout_01,4,3,2,5
2,0.5,1.5,20260908_013309_layout_02,8,6,6,12
3,0.5,1.5,20260908_013309_layout_03,3,3,4,7
4,0.5,1.5,20260908_013309_layout_04,5,3,3,8
...,...,...,...,...,...,...,...
715,4.0,3.0,20260908_013309_layout_35,2,1,1,1
716,4.0,3.0,20260908_013309_layout_36,1,1,1,1
717,4.0,3.0,20260908_013309_layout_37,1,1,1,1
718,4.0,3.0,20260908_013309_layout_38,2,1,1,1


In [31]:
summary = (runs.groupby(["PerceptionValue", "MaxSpeed"])
                .agg(n=("file", "size"),
                     Groups_start=("GroupsStart", "mean"),
                     Groups_x1=("GroupsX1", "mean"),
                     Groups_x2=("GroupsX2", "mean"),
                     Groups_end=("GroupsEnd", "mean"),
                     reached_x1=("GroupsX1", "count"),
                     reached_x2=("GroupsX2", "count"))
                .round(2)
                .reset_index())
summary

,PerceptionValue,MaxSpeed,n,Groups_start,Groups_x1,Groups_x2,Groups_end,reached_x1,reached_x2
0,0.5,1.5,40,4.58,3.98,4.28,8.68,40,40
1,0.5,3.0,40,7.60,13.85,14.02,18.10,40,40
2,1.0,1.5,40,5.85,5.85,5.88,7.30,40,40
3,1.0,3.0,40,6.22,6.62,6.75,9.45,40,40
4,1.5,1.5,40,3.52,3.50,3.40,3.75,40,40
5,1.5,3.0,40,3.22,3.32,3.38,6.28,40,40
6,1.8,1.5,40,3.22,2.58,2.70,3.00,40,40
7,1.8,3.0,40,2.82,2.62,2.65,5.30,40,40
8,2.0,1.5,40,3.30,2.17,2.17,2.50,40,40
9,2.0,3.0,40,2.90,2.42,2.50,4.92,40,40


In [37]:
r = runs.copy()
r["Split"]          = r["GroupsX2"] > r["GroupsX1"]
r["Merged"]         = r["GroupsX2"] < r["GroupsX1"]
r["Unchanged"]      = r["GroupsX2"] == r["GroupsX1"]
r["NeverMergedBack"] = r["GroupsEnd"] > r["GroupsX1"]
r["EndVsStart_More"] = r["GroupsEnd"] > r["GroupsStart"]

counts = (r.groupby(["PerceptionValue", "MaxSpeed"])
           .agg(
               #  reached_x2=("GroupsX2", "count"),
                Split=("Split", "sum"),
                Merged=("Merged", "sum"),
                Unchanged=("Unchanged", "sum"), n=("file", "size"))
                # NeverMergedBack=("NeverMergedBack", "sum"))
               #  MoreGroupsThanStart=("EndVsStart_More", "sum"))
           .reset_index())
counts

,PerceptionValue,MaxSpeed,Split,Merged,Unchanged,n
0,0.5,1.5,10,4,26,40
1,0.5,3.0,13,7,20,40
2,1.0,1.5,6,7,27,40
3,1.0,3.0,9,7,24,40
4,1.5,1.5,2,6,32,40
5,1.5,3.0,4,3,33,40
6,1.8,1.5,3,0,37,40
7,1.8,3.0,3,2,35,40
8,2.0,1.5,0,0,40,40
9,2.0,3.0,3,0,37,40


In [ ]:
def list_runs(perception=None, maxspeed=None, category=None, show_video=False):
    """category: "Split" | "Merged" | "Unchanged" | "NeverMergedBack" | "EndVsStart_More"
       (or a list of them -> AND).  None on any argument = no filter on it."""
    sel = r
    if perception is not None:
        sel = sel[sel["PerceptionValue"].round(4).isin(np.round(np.atleast_1d(perception), 4))]
    if maxspeed is not None:
        sel = sel[sel["MaxSpeed"].round(4).isin(np.round(np.atleast_1d(maxspeed), 4))]
    if category is not None:
        for c in np.atleast_1d(category):
            sel = sel[sel[c]]

    cols = ["file", "Layout", "PerceptionValue", "MaxSpeed",
            "GroupsStart", "GroupsX1", "GroupsX2", "GroupsEnd",
            "Split", "Merged", "Unchanged", "NeverMergedBack"]
    sel = (sel[cols]
             .sort_values(["PerceptionValue", "MaxSpeed", "Layout"])
             .reset_index(drop=True))

    print(f"{len(sel)} runs")
    for fn in sel["file"]:
        if show_video:
            fn = fn.replace(".json", ".mp4")
        print(os.path.join(RECORDING_DIR, fn))
    return sel


list_runs(perception=4, maxspeed=1.5, category="Merged", show_video=True)

0 runs


,file,Layout,PerceptionValue,MaxSpeed,GroupsStart,GroupsX1,GroupsX2,GroupsEnd,Split,Merged,Unchanged,NeverMergedBack


In [34]:
per_run.to_csv(f"per_run_groups_eps{EPS}.csv", index=False)
summary.to_csv(f"summary_groups_eps{EPS}.csv", index=False)
counts.to_csv(f"split_merge_counts_eps{EPS}.csv", index=False)

In [35]:
EPS_GRID    = [0.6, 0.8, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0]
SWEEP_EVERY = 20   # sample every Nth run

sweep_files = files[::SWEEP_EVERY]
sweep_rows  = Parallel(n_jobs=N_JOBS, verbose=5)(
    delayed(process_run)(f, eps=e) for e in EPS_GRID for f in sweep_files)

sweep = pd.DataFrame(sweep_rows)
sweep.pivot_table(index="eps",
                  values=["GroupsStart", "GroupsX1", "GroupsX2", "GroupsEnd"],
                  aggfunc="mean").round(2)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done 108 tasks      | elapsed:    1.0s
[Parallel(n_jobs=-1)]: Done 288 out of 288 | elapsed:    2.0s finished


,GroupsEnd,GroupsStart,GroupsX1,GroupsX2
eps,,,,
0.60,12.50,37.50,14.56,15.61
0.80,8.25,27.22,7.72,8.69
1.00,6.33,13.67,4.94,5.11
1.25,5.14,5.86,3.58,3.72
1.50,4.33,2.64,2.75,2.78
2.00,3.61,1.47,2.08,2.06
2.50,3.25,1.08,1.78,1.86
3.00,2.94,1.00,1.67,1.64
